In [14]:
import zipfile

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler


pd.options.display.max_columns = 300
pd.options.display.max_rows = 300
pd.options.display.max_colwidth = 300

X_train = pd.read_csv('train_data/dataset_1/data.csv')
y_train = pd.read_csv('train_data/y_train_base.csv')
X_test = pd.read_csv('train_data/X_test.csv')

display(
    'Тренировочные данные',
    X_train.shape,
    X_train.columns,
    'Тренировочные цены',
    y_train.shape,
    y_train.columns,
    'Тестовые данные',
    X_test.shape,
    X_test.columns,

)

'Тренировочные данные'

(174, 22)

Index(['car_id', 'Бренд', 'Год выпуска', 'Модель', 'Тип машины',
       'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод',
       'Топливо', 'Расход', 'Пробег', 'Цвет', 'Локация',
       'Количество цилиндров', 'Тип кузова', 'Двери', 'Количество кресел',
       'Оценка эксперта', 'Количество владельцев', 'Предложение'],
      dtype='object')

'Тренировочные цены'

(8340, 2)

Index(['car_id', 'Цена'], dtype='object')

'Тестовые данные'

(8341, 19)

Index(['car_id', 'Бренд', 'Год выпуска', 'Модель', 'Тип машины',
       'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод',
       'Топливо', 'Расход', 'Пробег', 'Цвет', 'Локация',
       'Количество цилиндров', 'Тип кузова', 'Двери', 'Количество кресел'],
      dtype='object')

In [15]:
# Проверка
X_test[X_test['car_id'].isin(y_train['car_id'].unique())].shape

(0, 19)

In [16]:
# Добавляем данные о цене в тренировачную выборку
print(X_train.shape)
print(y_train.shape)
X_train_with_price = X_train.merge(
    y_train,
    on='car_id'
)
print(X_train_with_price.shape)

(174, 22)
(8340, 2)
(174, 23)


In [17]:
# Делим на тренировачную и валидационную выборку
X = X_train_with_price[['Год выпуска', 'Пробег']]
y = X_train_with_price['Цена']
X.loc[:, 'Пробег'] = pd.to_numeric(X['Пробег'], errors='coerce').fillna(0)
# Разделяем на обучающую и валидационную выборки
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


In [18]:
# Создаем и обучаем модель
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [19]:
# Делаем предсказания
y_pred = model.predict(X_valid[['Год выпуска', 'Пробег']])

# Вычисляем метрики на валидационной выборке
mse = mean_absolute_error(y_valid, y_pred)

print(f"\nMAE: {mse:.2f}")


MAE: 21202.83


In [ ]:
# Предсказание на тестовых данных
# Предобработка столбца пробег
X_test.loc[:, 'Пробег'] = pd.to_numeric(X_test['Пробег'], errors='coerce').fillna(0)
y_test = model.predict(X_test[['Год выпуска', 'Пробег']])
y_test = pd.Series(y_test, name='Цена')

y_test.to_csv('submission.csv', index=False)

# Архивируем в submission.zip
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('submission.csv')

# 3 Случайное предсказание

In [ ]:
# y_test как Series со случайными значениями (Цена)
y_test = pd.Series(np.random.randint(0, 1000000, size=len(X_test)), name='Цена')

# Сохраняем в submission.csv
y_test.to_csv('submission.csv', index=False)

# Архивируем в submission.zip
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('submission.csv')